### 0) 임포트 & 전역 설정

In [0]:
# %%
# PySpark
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Pandas & NumPy
import pandas as pd
import numpy as np

# 회귀분석
import statsmodels.api as sm

# 시각화 (옵션)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rc

import time
from typing import List, Dict, Tuple

# 한글 폰트 (Databricks 노트북에서는 폰트 부재 가능)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)

print('모듈 임포트 완료')

### 1) 설정값 & 분석 대상 피처

In [0]:
# %%
# Spark 원본 테이블
SOURCE_TABLE = 'sandbox.z_jungryo_lee.agg_buzzmetrix_model_category_v1'

# 결과 저장 위치(Spark Delta 테이블)
# 출력 테이블/경로 이름
OUT_MODEL_TEST  = "sandbox.z_jungryo_lee.wls_model_summary_test"
OUT_COEF_TEST   = "sandbox.z_jungryo_lee.wls_coefficients_test"
OUT_MODEL_FULL  = "sandbox.z_jungryo_lee.wls_model_summary"
OUT_COEF_FULL   = "sandbox.z_jungryo_lee.wls_coefficients"

# 테스트 모드 설정
TEST_CONFIG = {
    "enabled": True,         # True면 테스트 실행 → 성공 시 전체 실행
    "targets_n": 3,          # 테스트에서 사용할 타깃(Y) 개수
    "groups_per_dim": 2,     # 각 그룹 차원별로 몇 개의 그룹값만 테스트
    "rows_per_group": 300,   # 각 그룹에서 최대 몇 개 model_id만 사용
    "random_seed": 42
}

# 결과 테이블 필터 기준(필요시 조정; 기본은 전체 기록)
APPLY_FILTERS = False  # True이면 아래 임계치로 필터
PVAL_THRESH   = 0.05
ABS_COEF_MIN  = 0.095
ABS_T_MIN     = 1.96

# 분석 대상 스마트 피처(카테고리 원본명)
smart_features = [
    '07-01. 채널/컨텐츠 APP',
    '07-02. 구동성/구동속도_(1)TV 전반',
    '07-02. 구동성/구동속도_(2)APP/웹전반',
    '07-03. 메뉴 구성/UI',
    '07-04. SW/OS',
    '07-05. 컨텐츠 탐색 사용성',
    '07-06. 리모컨 사용성',
    '07-07. 리모컨 디자인',
    '07-08. 음성 인식/조작',
    '07-09. 게임 기능',
    '07-10. 부가 기능',
    '07-11. 홈 IoT',
    '07-12. 모바일 연동',
    '07-13. 광고',
    '07-20. 전반적 스마트 사용성'
]

# 그룹 스펙: category1 / by 컬럼
group_specs = [
    {'category1': 'all',          'by': []},
    {'category1': 'brand_name',   'by': ['brand_name']},
    {'category1': 'year',         'by': ['year']},
    {'category1': 'country',      'by': ['country']},
    {'category1': 'd_type',       'by': ['d_type']},

]

print("설정 로드 완료")

### 2) Long→Wide 빌더 & WLS 함수

In [0]:
# %%
def clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
    new_cols = []
    for col in df.columns:
        c = str(col).replace('.', '').replace('/', '_').replace('(', '').replace(')', '').replace('-', '_').replace(' ', '_')
        new_cols.append(c)
    df.columns = new_cols
    return df


def build_wide_from_long(df_sub: pd.DataFrame,
                         id_name='model_id') -> pd.DataFrame:
    if id_name not in df_sub.columns:
        df_sub = build_model_id(df_sub, id_name=id_name)

    df_score = df_sub.pivot_table(index=id_name, columns='category', values='avg_sc').add_suffix('_score')
    df_count = df_sub.pivot_table(index=id_name, columns='category', values='total_count').add_suffix('_count')
    df_wide  = pd.concat([df_score, df_count], axis=1)
    df_wide.fillna(0, inplace=True)
    df_wide = clean_col_names(df_wide)
    return df_wide

def safe_wls(df_wide_subset: pd.DataFrame,
             target: str,
             all_features: List[str],
             apply_filters: bool = APPLY_FILTERS
            ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    반환: (model_df, coef_df)
      - model_df: 1행  ['y_spec','r_square','adj_r_square','model_p_value','y_obs']
      - coef_df: 다행 ['y_spec','x_spec','coef','p_value','t_value','x_obs']
    """
    t_score = f"{target}_score"
    t_count = f"{target}_count"

    empty_model = pd.DataFrame(columns=['y_spec','r_square','adj_r_square','model_p_value','y_obs'])
    empty_coef  = pd.DataFrame(columns=['y_spec','x_spec','coef','p_value','t_value','x_obs'])

    # 필수 컬럼 체크
    for col in (t_score, t_count):
        if col not in df_wide_subset.columns:
            return empty_model, empty_coef

    # X 후보(score 컬럼들)
    pred_cols = [f"{v}_score" for v in all_features if v != target and f"{v}_score" in df_wide_subset.columns]
    if len(pred_cols) == 0:
        return empty_model, empty_coef

    # 유효 표본: 타깃 가중치>0
    y = df_wide_subset[t_score]
    w = df_wide_subset[t_count]
    X = df_wide_subset[pred_cols]

    valid = w > 0
    if valid.sum() < 3:
        return empty_model, empty_coef

    y = y[valid]
    w = w[valid]
    X = X.loc[valid]

    # ---- (추가) X별 x_obs 계산: 같은 valid 표본 내에서 각 X의 *_count > 0 개수
    # pred_cols 기준으로 대응하는 count 컬럼명 만들기
    x_count_cols_all = {x_col: x_col.replace('_score', '_count') for x_col in X.columns}
    # 존재하는 count 컬럼만 골라서 벡터화 계산
    existing_count_cols = [c for c in x_count_cols_all.values() if c in df_wide_subset.columns]
    if existing_count_cols:
        x_counts_df = df_wide_subset.loc[valid, existing_count_cols]
        x_obs_series = (x_counts_df > 0).sum(axis=0).astype(int)  # 각 count컬럼별 True 개수
    else:
        x_obs_series = pd.Series(dtype=int)

    # 분산 0 예측변수 제거(상수열)
    non_const = X.std(numeric_only=True) > 0
    X = X.loc[:, non_const]
    if X.shape[1] == 0:
        return empty_model, empty_coef

    # 상수항 추가하고 적합
    Xc = sm.add_constant(X, has_constant='add')
    try:
        model = sm.WLS(y, Xc, weights=w).fit()
    except Exception:
        return empty_model, empty_coef

    # 모델 지표 및 y_obs
    r2     = float(getattr(model, 'rsquared', np.nan))
    r2_adj = float(getattr(model, 'rsquared_adj', np.nan))
    f_pval = float(getattr(model, 'f_pvalue', np.nan))
    y_obs  = int(getattr(model, 'nobs', len(y)))

    model_df = pd.DataFrame([{
        'y_spec'        : target,
        'r_square'      : r2,
        'adj_r_square'  : r2_adj,
        'model_p_value' : f_pval,
        'y_obs'         : y_obs
    }], columns=['y_spec','r_square','adj_r_square','model_p_value','y_obs'])

    # 계수(상수항 제외)
    params  = model.params.drop(labels=['const'], errors='ignore')
    pvalues = model.pvalues.drop(labels=['const'], errors='ignore')
    tvalues = model.tvalues.drop(labels=['const'], errors='ignore')

    rows = []
    for x_col in params.index:
        coef = float(params[x_col])
        pv   = float(pvalues.get(x_col, np.nan))
        tv   = float(tvalues.get(x_col, np.nan))

        # 이 X의 count 컬럼명
        c_count = x_col.replace('_score', '_count')
        # 벡터화로 계산했던 x_obs 가져오기 (없으면 유효표본 수로 fallback)
        x_obs = int(x_obs_series.get(c_count, y_obs))

        rec = {
            'y_spec'  : target,
            'x_spec'  : x_col.replace('_score', ''),
            'coef'    : coef,
            'p_value' : pv,
            't_value' : tv,
            'x_obs'   : x_obs
        }

        if apply_filters:
            if (abs(coef) >= ABS_COEF_MIN) and (abs(tv) >= ABS_T_MIN) and (pv <= PVAL_THRESH):
                rows.append(rec)
        else:
            rows.append(rec)

    coef_df = pd.DataFrame(rows, columns=['y_spec','x_spec','coef','p_value','t_value','x_obs'])

    return model_df, coef_df


### 3) Spark → Pandas(long) → Wide + key 매핑

In [0]:
# %%
# 1) Spark → Pandas
df_spark = spark.table(SOURCE_TABLE)

# 필요 시 Spark에서 선필터(예: 최근 연도/특정 국가) → 메모리 절감
# df_spark = df_spark.filter(F.col("year") >= 2022)

df_pandas = df_spark.toPandas()
print(f"[LOAD] Rows: {len(df_pandas):,}")
display(df_pandas.head(3))

# 2) Long 준비
df_long = df_pandas.copy()
df_long['model_id'] = (
    df_long['country'].astype(str) + '_' + df_long['year'].astype(str) + '_' + df_long['model'].astype(str)
)

# 3) Wide 변환
df_wide = build_wide_from_long(df_long, id_name='model_id')

# 4) key 매핑 (model_id → 그룹키들)
df_key = (
    df_long[['model_id', 'country', 'year', 'brand_name', 'd_type', 'model']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("[WIDE] 샘플")
display(df_wide.reset_index().head(3))
print("[KEY] 샘플")
display(df_key.head(3))


### 4) 그룹 실행 유틸 (테스트/전체)

In [0]:
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
from datetime import datetime
import json

def run_groups(df_wide: pd.DataFrame,
               df_key: pd.DataFrame,
               group_specs: List[Dict],
               smart_features: List[str],
               test_mode: bool = False,
               test_config: Dict = None
              ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    반환: (unified_model_df, unified_coef_df, test_raw_df)
      - unified_model_df: ['category1','category2','y_spec','r_square','adj_r_square','model_p_value','y_obs']
      - unified_coef_df : ['category1','category2','y_spec','x_spec','coef','p_value','t_value','x_obs']
      - test_raw_df     : 테스트 모드에서 사용된 원본 샘플 (df_wide_sub) + 메타
                          ['run_id','run_ts','category1','category2','y_spec','model_id', <df_wide 컬럼들...>]
    """
    if test_mode and test_config is None:
        test_config = {"targets_n": 3, "groups_per_dim": 2, "rows_per_group": 300, "random_seed": 42}

    rng = np.random.default_rng(test_config.get("random_seed", 42)) if test_mode else None
    run_ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    run_id = f"test_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{test_config.get('random_seed', 'na')}"
    test_blocks = []  # <- 여기 누적

    targets = smart_features 
    if test_mode:
        k = min(test_config["targets_n"], len(smart_features))
        targets = list(rng.choice(smart_features, size=k, replace=False))

    model_blocks = []
    coef_blocks  = []
    all_model_ids = set(df_wide.index)

    for gspec in group_specs:
        cat1 = gspec['category1']
        by   = gspec['by']

        if len(by) == 0:
            # 전체(all)
            df_wide_sub = df_wide
            if test_mode:
                idx = df_wide_sub.index.to_list()
                if len(idx) > test_config["rows_per_group"]:
                    idx = list(rng.choice(idx, size=test_config["rows_per_group"], replace=False))
                df_wide_sub = df_wide_sub.loc[idx]

                # 테스트 입력 기록
                tmp = df_wide_sub.copy()
                tmp.insert(0, 'model_id', tmp.index)  # 인덱스가 model_id 라고 가정
                tmp.insert(0, 'y_spec', None)         # target별로 별도 복제
                tmp.insert(0, 'category2', 'all')
                tmp.insert(0, 'category1', cat1)
                tmp.insert(0, 'run_ts', run_ts)
                tmp.insert(0, 'run_id', run_id)
                # target별로 기록 필요하면 아래에서 복제
                # 일단은 이 시점에서 raw만 저장하고, target은 루프에서 채움

            for target in targets:
                model_df, coef_df = safe_wls(df_wide_sub, target, smart_features, apply_filters=APPLY_FILTERS)
                if not model_df.empty:
                    model_df.insert(0, 'category2', 'all')
                    model_df.insert(0, 'category1', cat1)
                    model_blocks.append(model_df)
                if not coef_df.empty:
                    coef_df.insert(0, 'category2', 'all')
                    coef_df.insert(0, 'category1', cat1)
                    coef_blocks.append(coef_df)

                if test_mode:
                    # target별로 한 번 더 붙여서 기록(동일 raw에 y_spec만 표기)
                    ttmp = tmp.copy()
                    ttmp['y_spec'] = target
                    test_blocks.append(ttmp)

        else:
            grp = df_key[by].drop_duplicates()
            if test_mode and len(grp) > test_config["groups_per_dim"]:
                grp = grp.sample(n=test_config["groups_per_dim"], random_state=test_config["random_seed"])

            for _, row in grp.iterrows():
                if len(by) == 1:
                    cat2 = str(row[by[0]])
                    ids = set(df_key.loc[df_key[by[0]] == row[by[0]], 'model_id'])
                else:
                    cat2 = "|".join(str(row[c]) for c in by)
                    mask = np.ones(len(df_key), dtype=bool)
                    for c in by:
                        mask &= (df_key[c] == row[c]).to_numpy()
                    ids = set(df_key.loc[mask, 'model_id'])

                ids = [mid for mid in ids if mid in all_model_ids]
                if not ids:
                    continue

                df_wide_sub = df_wide.loc[ids]
                if test_mode and len(df_wide_sub) > test_config["rows_per_group"]:
                    idx = list(rng.choice(df_wide_sub.index.to_list(), size=test_config["rows_per_group"], replace=False))
                    df_wide_sub = df_wide_sub.loc[idx]

                # 테스트 입력 저장(타겟 넣기 전 베이스 프레임)
                if test_mode:
                    base_tmp = df_wide_sub.copy()
                    base_tmp.insert(0, 'model_id', base_tmp.index)
                    base_tmp.insert(0, 'y_spec', None)
                    base_tmp.insert(0, 'category2', cat2)
                    base_tmp.insert(0, 'category1', cat1)
                    base_tmp.insert(0, 'run_ts', run_ts)
                    base_tmp.insert(0, 'run_id', run_id)

                for target in targets:
                    model_df, coef_df = safe_wls(df_wide_sub, target, smart_features, apply_filters=APPLY_FILTERS)
                    if not model_df.empty:
                        model_df.insert(0, 'category2', cat2)
                        model_df.insert(0, 'category1', cat1)
                        model_blocks.append(model_df)
                    if not coef_df.empty:
                        coef_df.insert(0, 'category2', cat2)
                        coef_df.insert(0, 'category1', cat1)
                        coef_blocks.append(coef_df)

                    if test_mode:
                        ttmp = base_tmp.copy()
                        ttmp['y_spec'] = target
                        test_blocks.append(ttmp)

    if model_blocks:
        unified_model_df = pd.concat(model_blocks, ignore_index=True)
    else:
        unified_model_df = pd.DataFrame(columns=[
            'category1','category2','y_spec','r_square','adj_r_square','model_p_value','y_obs'
        ])

    if coef_blocks:
        unified_coef_df = pd.concat(coef_blocks, ignore_index=True)
    else:
        unified_coef_df = pd.DataFrame(columns=[
            'category1','category2','y_spec','x_spec','coef','p_value','t_value','x_obs'
        ])

    # 테스트 원본 묶기
    if test_blocks:
        test_raw_df = pd.concat(test_blocks, ignore_index=True)
        # test_config를 컬럼으로 남기고 싶으면 문자열화
        test_raw_df['test_config_json'] = json.dumps(test_config, ensure_ascii=False)
    else:
        test_raw_df = pd.DataFrame(columns=['run_id','run_ts','category1','category2','y_spec','model_id'])

    # 컬럼 순서 정리
    unified_model_df = unified_model_df[
        ['category1','category2','y_spec','r_square','adj_r_square','model_p_value','y_obs']
    ]
    unified_coef_df  = unified_coef_df[
        ['category1','category2','y_spec','x_spec','coef','p_value','t_value','x_obs']
    ]
    # test_raw_df는 df_wide 컬럼 구성이 다양하므로 고정하지 않음

    return unified_model_df, unified_coef_df, test_raw_df

In [0]:
def basic_sanity_checks_two(df_model: pd.DataFrame, df_coef: pd.DataFrame, phase: str):
    req_model = {'category1','category2','y_spec','r_square','adj_r_square','model_p_value','y_obs'}
    req_coef  = {'category1','category2','y_spec','x_spec','coef','p_value','t_value','x_obs'}

    miss_m = req_model - set(df_model.columns)
    miss_c = req_coef  - set(df_coef.columns)
    if miss_m:
        raise AssertionError(f"[{phase}] 모델 테이블 필수 컬럼 누락: {miss_m}")
    if miss_c:
        raise AssertionError(f"[{phase}] 계수 테이블 필수 컬럼 누락: {miss_c}")

    if len(df_model) == 0:
        print(f"[{phase}] 모델 테이블이 비어있습니다.")
    if len(df_coef) == 0:
        print(f"[{phase}] 계수 테이블이 비어있습니다.")

    # 정보 출력
    if len(df_model):
        na_ratio = df_model[['r_square','adj_r_square','model_p_value']].isna().mean().round(3)
        print(f"[{phase}] 모델 지표 NaN 비율:\n{na_ratio}")

def save_two_outputs(model_df: pd.DataFrame, coef_df: pd.DataFrame,
                     model_table: str, coef_table: str
                     ):
    # Spark 테이블
    spark.createDataFrame(model_df).write.mode("overwrite").saveAsTable(model_table)
    spark.createDataFrame(coef_df ).write.mode("overwrite").saveAsTable(coef_table)
    print(f"[SAVE] Spark 저장 완료: {model_table}, {coef_table}")


In [0]:
# --- TEST ---
tests_passed = True
if TEST_CONFIG["enabled"]:
    print("=== [TEST] 부분 실행 시작 ===")
    try:
        m_test, c_test, test_raw_df = run_groups(
            df_wide=df_wide, df_key=df_key, group_specs=group_specs,
            smart_features=smart_features, test_mode=True, test_config=TEST_CONFIG
        )
        display(m_test.head(5)); display(c_test.head(5))
        print(f"[TEST] model rows: {len(m_test):,}, coef rows: {len(c_test):,}")
        basic_sanity_checks_two(m_test, c_test, phase="TEST")
        save_two_outputs(
            m_test, c_test,
            OUT_MODEL_TEST, OUT_COEF_TEST
        )
    except Exception as e:
        tests_passed = False
        print("[TEST] 에러 발생 → 전체 실행 중단")
        raise
    finally:
        print("=== [TEST] 완료 ===")
else:
    print("[TEST] 비활성화")

# --- FULL ---
# if (not TEST_CONFIG["enabled"]) or tests_passed:
#     print("\n=== [FULL] 전체 실행 시작 ===")
#     try:
#         m_full, c_full = run_groups(
#             df_wide=df_wide, df_key=df_key, group_specs=group_specs,
#             smart_features=smart_features, test_mode=False
#         )
#         display(m_full.head(5)); display(c_full.head(5))
#         print(f"[FULL] model rows: {len(m_full):,}, coef rows: {len(c_full):,}")
#         basic_sanity_checks_two(m_full, c_full, phase="FULL")
#         save_two_outputs(
#             m_full, c_full,
#             OUT_MODEL_FULL, OUT_COEF_FULL,
#             CSV_MODEL_FULL,  CSV_COEF_FULL,
#             XLSX_MODEL_FULL, XLSX_COEF_FULL
#         )
#     except Exception as e:
#         print("[FULL] 에러 발생")
#         raise
#     finally:
#         print("=== [FULL] 완료 ===")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# run_id 기준으로 파티션 저장하고 싶으면:
from pyspark.sql import functions as F
sdf = spark.createDataFrame(clean_col_names(test_raw_df))

RUN_TABLE = "sandbox.z_jungryo_lee.test_wls_input"
# run_id가 컬럼에 이미 있음. 파티션 저장:
(sdf
 .write
 .mode("append")       # 여러 번 실행 시 누적
 .format("delta")
 .partitionBy("run_id")  # 실행별 구분/증분조회 용이
 .saveAsTable(RUN_TABLE)
)


In [0]:
display(sdf)